In [1]:
import torch

if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.is_available()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory Total: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)} GB")
    device = torch.device("cuda:0")
else:
    print("No GPU available, training on CPU.")
    device = torch.device("cpu")


GPU is available: True
GPU Name: NVIDIA RTX 6000 Ada Generation
GPU Memory Total: 47.99 GB


In [2]:
# !pip install surfa trimesh nibabel numpy torch

In [3]:
# # import torch
# TORCH = torch.__version__.split('+')[0]
# CUDA = 'cu' + torch.version.cuda.replace('.', '')

# # 2. Install torch_scatter via the official wheel
# !pip install torch-scatter -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html


In [4]:
# !pip install torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
# !pip install torch-cluster -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

In [5]:
import torch
import numpy as np
import surfa as sf
import os
from topofit.topofit import ico, io
from model import AttentionBasedEncoder # <-- Updated Class Name!
from layers.mesh import Mesh # Import YOUR custom MeshCNN object!

In [6]:
class DummyOpt:
    """MeshCNN requires an options object to initialize. We provide a dummy one here."""
    num_aug = 1
    # Add any other default MeshCNN arguments your mesh.py might check for

def convert_to_meshcnn(surfa_template, save_path="clean_lh.obj"):
    """Converts a surfa.Mesh into a layers.mesh.Mesh by bridging via .obj format"""
    print("Building MeshCNN Topological Edges (This takes a moment...)")

    # 1. Write the surfa vertices and faces to an .obj file
    with open(save_path, 'w') as f:
        for v in surfa_template.vertices:
            f.write(f"v {v[0]} {v[1]} {v[2]}\n")
        for face in surfa_template.faces:
            # .obj files are 1-indexed, so we add 1 to the face indices
            f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")

    # 2. Load it into your custom MeshCNN object so it builds the gemm_edges
    opt = DummyOpt()
    meshcnn_obj = Mesh(file=save_path, opt=opt)
    return meshcnn_obj

# --- 4. The Left Hemisphere Loader ---
def load_left_hemisphere(subj_dir):
    print("Loading MRI and aligning Left Hemisphere template...")
    image = sf.load_volume(f'{subj_dir}/mri/norm.mgz')
    affine = sf.load_affine(f'{subj_dir}/mri/transforms/talairach.xfm.lta').inv().convert(space='vox', target=image)

    # Get the TopoFit template
    template = ico.get_initial_template('lh')
    template.vertices = affine.transform(template.vertices)

    crop_idx = io.compute_image_cropping(image.baseshape, template.vertices)
    cropped_img = image[crop_idx].reshape((96, 144, 192))
    cropped_img = cropped_img.astype(np.float32) / cropped_img.percentile(99.99, nonzero=True)

    verts = template.convert(space='vox', geometry=cropped_img).vertices.astype(np.float32)
    template.vertices = verts # Update the template with cropped vertices

    # --- NEW: Convert to MeshCNN Object ---
    meshcnn_topology = convert_to_meshcnn(template)

    mri_tensor = torch.tensor(cropped_img.data).unsqueeze(0).unsqueeze(0)
    vert_tensor = torch.tensor(verts).unsqueeze(0)

    return mri_tensor, vert_tensor, [meshcnn_topology]

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

subj_path = "/mnt/c/Users/ADMIN/Documents/Dev_Projects/brain mri/VisionEncoder/OASIS_0001"
mri_batch, vertex_batch, mesh_topology = load_left_hemisphere(subj_path)

mri_batch = mri_batch.to(device, dtype=torch.float32)
vertex_batch = vertex_batch.to(device, dtype=torch.float32)

print(f"Input MRI Shape: {mri_batch.shape}")
print(f"Input Vertex Shape: {vertex_batch.shape}")

print("\nInitializing Architecture...")
model = AttentionBasedEncoder().to(device) # <-- Updated Class Name!

print("Running Forward Pass...")
with torch.no_grad():
    out_features, _ = model(mri_batch, vertex_batch, mesh_topology)

print(f"\nSUCCESS! Output Feature Shape: {out_features.shape}")

Using Device: cuda
Loading MRI and aligning Left Hemisphere template...
Building MeshCNN Topological Edges (This takes a moment...)
Input MRI Shape: torch.Size([1, 1, 96, 144, 192])
Input Vertex Shape: torch.Size([1, 40962, 3])

Initializing Architecture...
Running Forward Pass...

SUCCESS! Output Feature Shape: torch.Size([1, 40962, 62])
